# 02 - Tiền xử lý dữ liệu (ĐÃ SỬA LỖI RÒ RỈ DỮ LIỆU)
BTL môn Trí tuệ nhân tạo — Dữ liệu mô phỏng

**Thay đổi quan trọng so với bản cũ:** Chia Train/Test **TRƯỚC** khi chuẩn hóa dữ liệu số. `StandardScaler` chỉ được `fit()` trên tập Train, sau đó `transform()` cho cả Train và Test. Bản cũ `fit_transform()` trên toàn bộ X trước khi chia tập — đây là lỗi rò rỉ dữ liệu (data leakage) bị nêu rõ trong đề cương môn học.

## Bước 1: Đọc dữ liệu

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/du_lieu_sinh_vien_mophong.csv")
df.head()

,GioiTinh,KhuVucSinhSong,HocVanChaMe,HoanCanhGiaDinh,HocLucNamTruoc,TiLeChuyenCan,GioTuHocMoiTuan,ThamGiaHocThem,OnThiTruoc,ThoiGianMangXaHoi,KetQua
0,Nam,Thành thị,Tốt nghiệp THPT,Trung bình,Trung bình,94.4,11.1,Có,Đã ôn thi,2.0,Đỗ
1,Nam,Thành thị,Đại học,Khá giả,Khá,97.0,12.2,Có,Đã ôn thi,5.8,Đỗ
2,Nữ,Nông thôn,Đại học,Trung bình,Khá,84.5,10.7,Có,Đã ôn thi,3.8,Đỗ
3,Nam,Nông thôn,Đại học,Trung bình,Khá,73.1,6.8,Không,Đã ôn thi,5.1,Đỗ
4,Nữ,Thành thị,Thạc sĩ,Khó khăn,Trung bình,93.4,12.8,Không,Đã ôn thi,7.7,Trượt


## Bước 2: Kiểm tra dữ liệu thiếu

In [2]:
print(df.isnull().sum())

GioiTinh             0
KhuVucSinhSong       0
HocVanChaMe          0
HoanCanhGiaDinh      0
HocLucNamTruoc       0
TiLeChuyenCan        0
GioTuHocMoiTuan      0
ThamGiaHocThem       0
OnThiTruoc           0
ThoiGianMangXaHoi    0
KetQua               0
dtype: int64


## Bước 3: Tạo nhãn dạng số (Đỗ=1, Trượt=0)

In [3]:
df['Label'] = df['KetQua'].apply(lambda x: 1 if x == 'Đỗ' else 0)
print(df['Label'].value_counts())

Label
1    975
0    525
Name: count, dtype: int64


## Bước 4: Mã hóa dữ liệu dạng chữ (Label Encoding)
**Lưu ý về rò rỉ dữ liệu:** `LabelEncoder` chỉ ánh xạ tên danh mục (VD: 'Nam'→0, 'Nữ'→1) sang số, **không sử dụng bất kỳ thông tin nào về nhãn `Label`**, nên bước này an toàn để làm trên toàn bộ dữ liệu trước khi chia tập — khác với `StandardScaler` (dùng trung bình/độ lệch chuẩn — con số phụ thuộc vào việc mẫu nào nằm trong tập nào).

In [4]:
from sklearn.preprocessing import LabelEncoder

cat_cols = ['GioiTinh', 'KhuVucSinhSong', 'HocVanChaMe', 'HoanCanhGiaDinh',
            'HocLucNamTruoc', 'ThamGiaHocThem', 'OnThiTruoc']

encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

df.head()

,GioiTinh,KhuVucSinhSong,HocVanChaMe,HoanCanhGiaDinh,HocLucNamTruoc,TiLeChuyenCan,GioTuHocMoiTuan,ThamGiaHocThem,OnThiTruoc,ThoiGianMangXaHoi,KetQua,Label
0,0,1,3,2,2,94.4,11.1,0,1,2.0,Đỗ,1
1,0,1,4,0,1,97.0,12.2,0,1,5.8,Đỗ,1
2,1,0,4,2,1,84.5,10.7,0,1,3.8,Đỗ,1
3,0,0,4,2,1,73.1,6.8,1,1,5.1,Đỗ,1
4,1,1,2,1,2,93.4,12.8,1,1,7.7,Trượt,0


## Bước 5: Tách Features (X) và Label (y)

In [5]:
X = df.drop(columns=['KetQua', 'Label'])
y = df['Label']
feature_names = X.columns.tolist()

print(X.shape, y.shape)
print(feature_names)

(1500, 10) (1500,)
['GioiTinh', 'KhuVucSinhSong', 'HocVanChaMe', 'HoanCanhGiaDinh', 'HocLucNamTruoc', 'TiLeChuyenCan', 'GioTuHocMoiTuan', 'ThamGiaHocThem', 'OnThiTruoc', 'ThoiGianMangXaHoi']


## Bước 6: Chia Train/Test TRƯỚC khi chuẩn hóa ⚠️ (thứ tự đã sửa)
Đây là thay đổi cốt lõi: `train_test_split` được gọi trên `X` **thô** (chưa chuẩn hóa), không phải trên `X_scaled` như bản cũ.

In [6]:
from sklearn.model_selection import train_test_split

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train_raw.shape, "Test:", X_test_raw.shape)

Train: (1200, 10) Test: (300, 10)


## Bước 7: Chuẩn hóa — fit CHỈ trên Train, transform cho cả hai ✅
`scaler.fit_transform(X_train_raw)` — học tham số (mean, std) **chỉ từ dữ liệu Train**.

`scaler.transform(X_test_raw)` — áp dụng lại đúng tham số đó cho Test, **không fit lại**.

Nhờ vậy, tập Test hoàn toàn không ảnh hưởng đến quá trình chuẩn hóa — đúng nguyên tắc "chỉ đo hiệu năng trên dữ liệu độc lập hoàn toàn với huấn luyện".

In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)   # fit + transform trên Train
X_test = scaler.transform(X_test_raw)         # CHỈ transform trên Test

print("Đã chuẩn hóa xong, không còn rò rỉ dữ liệu qua bước scaling.")

Đã chuẩn hóa xong, không còn rò rỉ dữ liệu qua bước scaling.


## Bước 8: Lưu kết quả

In [8]:
import joblib
import os

os.makedirs("../data/processed", exist_ok=True)

joblib.dump((X_train, X_test, y_train, y_test), "../data/processed/train_test_data.pkl")
joblib.dump(scaler, "../data/processed/scaler.pkl")
joblib.dump(encoders, "../data/processed/encoders.pkl")
joblib.dump(feature_names, "../data/processed/feature_names.pkl")

print("Đã lưu xong dữ liệu đã xử lý (không rò rỉ)! Chuyển sang notebook 03_train_models.ipynb")

Đã lưu xong dữ liệu đã xử lý (không rò rỉ)! Chuyển sang notebook 03_train_models.ipynb
